In [4]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder, StandardScaler
import numpy as np

# Load dataset
df = pd.read_csv("../data/all_participants_features.csv")

# Features & target
X = df.drop(columns=["Participant", "TaskKey", "CognitiveLoad"])
y = df["CognitiveLoad"]

X.replace([np.inf, -np.inf], np.nan, inplace=True)  # convert inf → NaN
X.fillna(X.mean(), inplace=True)
X = X.clip(lower=-1e6, upper=1e6)

# Encode target if categorical
if y.dtype == "object":
    le = LabelEncoder()
    y = le.fit_transform(y)


In [5]:
binary = False  # Set True for binary classification

if binary:
    y = y.copy()
    y[y == 1] = 0  # Example: map Medium to Low
    y[y == 2] = 1  # High stays High


In [6]:
participants = df["Participant"].unique()

# Personalized models dictionary
personalized_models = {}

for p in participants:
    X_p = X[df["Participant"] == p]
    y_p = y[df["Participant"] == p]
    
    # Scale features
    scaler = StandardScaler()
    X_p_scaled = scaler.fit_transform(X_p)
    
    # Train your classifier here, e.g., RandomForest
    from sklearn.ensemble import RandomForestClassifier
    model = RandomForestClassifier(random_state=42)
    model.fit(X_p_scaled, y_p)
    
    personalized_models[p] = model


In [7]:
# Example: compute slope between consecutive time points
import numpy as np

for col in X.columns:
    X[col + "_slope"] = X[col].diff().fillna(0)


In [8]:
def sliding_window(X, y, window_size=5, step=1):
    X_windows, y_windows = [], []
    for start in range(0, len(X) - window_size + 1, step):
        end = start + window_size
        X_windows.append(X[start:end].values)
        y_windows.append(y[end-1])  # label of last in window
    return np.array(X_windows), np.array(y_windows)

X_windows, y_windows = sliding_window(X, y, window_size=5)


In [9]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, LSTM, Dense, Flatten, Dropout

model = Sequential()
model.add(Conv1D(filters=64, kernel_size=3, activation='relu', input_shape=(X_windows.shape[1], X_windows.shape[2])))
model.add(LSTM(64, return_sequences=False))
model.add(Dense(len(np.unique(y_windows)), activation='softmax'))

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.fit(X_windows, y_windows, epochs=20, batch_size=32, validation_split=0.2)


d:\IITB\venv\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/20
34/34 ━━━━━━━━━━━━━━━━━━━━ 8s 39ms/step - accuracy: 0.4347 - loss: 1.1597 - val_accuracy: 0.6029 - val_loss: 1.0876
Epoch 2/20
34/34 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.6489 - loss: 0.9280 - val_accuracy: 0.7059 - val_loss: 0.8420
Epoch 3/20
34/34 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.7886 - loss: 0.6490 - val_accuracy: 0.8493 - val_loss: 0.5943
Epoch 4/20
34/34 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.8676 - loss: 0.4471 - val_accuracy: 0.8897 - val_loss: 0.4565
Epoch 5/20
34/34 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.8869 - loss: 0.3407 - val_accuracy: 0.8824 - val_loss: 0.3690
Epoch 6/20
34/34 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.9412 - loss: 0.2365 - val_accuracy: 0.9301 - val_loss: 0.2810
Epoch 7/20
34/34 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.9522 - loss: 0.1823 - val_accuracy: 0.9301 - val_loss: 0.2485
Epoch 8/20
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.9623 - loss: 0.1474 - val_accuracy: 0.9412 - v